Notebook 1 — EDA & Cleaning

In [20]:
import pandas as pd 
from pathlib import Path

DATA_PATH = Path('../data/climate_change_impact_on_agriculture_2024.csv')
df = pd.read_csv(DATA_PATH)

print("=" * 50)
print("DATASET OVERVIEW")
print('=' * 50)
print(f"Shape: {df.shape[0]:,} rows * {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColumn types:\n{df.dtypes.value_counts()}")
print(f"\nThe data types of the columns of the table: \n{df.dtypes}")
print(f"\nFirst 3 rows:\n{df.head(3)}")

DATASET OVERVIEW
Shape: 10,000 rows * 15 columns
Memory: 3.08 MB

Column types:
float64    9
object     4
int64      2
Name: count, dtype: int64

The data types of the columns of the table: 
Year                             int64
Country                         object
Region                          object
Crop_Type                       object
Average_Temperature_C          float64
Total_Precipitation_mm         float64
CO2_Emissions_MT               float64
Crop_Yield_MT_per_HA           float64
Extreme_Weather_Events           int64
Irrigation_Access_%            float64
Pesticide_Use_KG_per_HA        float64
Fertilizer_Use_KG_per_HA       float64
Soil_Health_Index              float64
Adaptation_Strategies           object
Economic_Impact_Million_USD    float64
dtype: object

First 3 rows:
   Year Country         Region Crop_Type  Average_Temperature_C  \
0  2001   India    West Bengal      Corn                   1.55   
1  2024   China          North      Corn                   3.

Cell 2 — Missing values

Checking NaN count per column
Checking percentage missing
Handle any missing values appropriately

In [21]:
def missing_value_report(df):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    report = pd.DataFrame({
        "Missing Counts": missing,
        "Missing %": missing_pct,
        "Dtype": df.dtypes
    }).sort_values("Missing %", ascending=False)
    
    report = report[report["Missing Counts"] > 0]
    
    if len(report) == 0:
        print("No missing values found")
    else:
        df.fillna("Unknown", inplace=True)
        print("Missing values are filled with 'Unknown")
        print(report)
    
    return report

missing_report = missing_value_report(df)

No missing values found


Cell 3 — Explore key columns

How many unique countries?
How many unique crop types?
What years does the data cover?
How many unique regions?

In [22]:
def dataset_summary(df):
    print("=" * 50)
    print("DATASET SUMMARY")
    print("=" * 50)
    
    cat_cols = df.select_dtypes(include="object").columns
    
    for col in cat_cols:
        print(f"\n{col} ({df[col].nunique()} unique):")
        print(f" {sorted(df[col].unique())}")
    
    print(f"\nYear range: {df['Year'].min()} -> {df['Year'].max()}")
    print(f"Total years: {df['Year'].nunique()}")
    print(f"\nNumerical summary")
    print(df.describe().round(2))

dataset_summary(df)

DATASET SUMMARY

Country (10 unique):
 ['Argentina', 'Australia', 'Brazil', 'Canada', 'China', 'France', 'India', 'Nigeria', 'Russia', 'USA']

Region (34 unique):
 ['British Columbia', 'Central', 'East', 'Grand Est', 'Ile-de-France', 'Maharashtra', 'Midwest', 'New South Wales', 'North', 'North Central', 'North West', 'Northeast', 'Northwest', 'Northwestern', 'Nouvelle-Aquitaine', 'Ontario', 'Pampas', 'Patagonia', 'Prairies', 'Provence-Alpes-Cote d’Azur', 'Punjab', 'Quebec', 'Queensland', 'Siberian', 'South', 'South East', 'South West', 'Southeast', 'Tamil Nadu', 'Victoria', 'Volga', 'West', 'West Bengal', 'Western Australia']

Crop_Type (10 unique):
 ['Barley', 'Coffee', 'Corn', 'Cotton', 'Fruits', 'Rice', 'Soybeans', 'Sugarcane', 'Vegetables', 'Wheat']

Adaptation_Strategies (5 unique):
 ['Crop Rotation', 'Drought-resistant Crops', 'No Adaptation', 'Organic Farming', 'Water Management']

Year range: 1990 -> 2024
Total years: 35

Numerical summary
           Year  Average_Temperature_C

Cell 4 — Feature engineering
Adding new columns:

In [24]:
def engineer_features(df):
    df = df.copy()
    
    df["Decade"] = (df["Year"] // 10) * 10
    
    #yield classification
    df["Yield_Category"] = pd.cut(
        df["Crop_Yield_MT_per_HA"],
        bins = [0, 2, 4, df["Crop_Yield_MT_per_HA"].max()],
        labels = ["Low", "Medium", "High"]
    )
    
    #setting stress indicator
    df["High_Stress"] = (
        (df["Extreme_Weather_Events"] >= 7) &
        (df["Soil_Health_Index"] < 50)
    )
    
    #temporal zones
    df["Temp_Zones"] = pd.cut(
        df["Average_Temperature_C"],
        bins = [-float("inf"), 10, 20, 30, float("inf")],
        labels = ["Cold", "Temperature", "Warm", "Hot"]
    )
    
    #water avaliability score
    df["Water_Score"] = (
        df["Total_Precipitation_mm"].rank(pct=True) * 0.6 +
        df["Irrigation_Access_%"].rank(pct=True) * 0.4
    ).round(4)
    
    print("Features are done successfully")
    print("New columns added: Decade, Yield_Category, High_Stress, Temp_Zones, Water_Score")
    print(f"\nYield distribution: \n{df['Yield_Category'].value_counts()}")
    print(f"\nHigh stress records: {df["High_Stress"].sum():,} ({df['High_Stress'].mean()*100:.1f}%)")
    
    return df

df= engineer_features(df)

Features are done successfully
New columns added: Decade, Yield_Category, High_Stress, Temp_Zones, Water_Score

Yield distribution: 
Yield_Category
Medium    5064
Low       4439
High       497
Name: count, dtype: int64

High stress records: 1,022 (10.2%)


Cell 5 — Crop types relevant to Central Asia

In [27]:
ca_crops = ["Wheat", "Cotton", "Rice", "Barley"]

ca_df = df[df["Crop_Type"].isin(ca_crops)].copy()

print("=" * 50)
print("CENTRAL ASIA RELEVANT CROPS SUBSET")
print("=" * 50)
print(f"Rows: {len(ca_df):,} / {len(df):,} ({len(ca_df)/len(df)*100:.1f}%)")
print(f"\nCrop distribution:")
print(ca_df["Crop_Type"].value_counts())
print(f"\nYield stats by crop:")
print(ca_df.groupby("Crop_Type")["Crop_Yield_MT_per_HA"].agg(["mean", "std", "min", "max"]).round(2))

CENTRAL ASIA RELEVANT CROPS SUBSET
Rows: 4,065 / 10,000 (40.6%)

Crop distribution:
Crop_Type
Wheat     1047
Cotton    1044
Rice      1022
Barley     952
Name: count, dtype: int64

Yield stats by crop:
           mean   std   min   max
Crop_Type                        
Barley     2.29  1.00  0.46  5.00
Cotton     2.17  0.97  0.48  4.91
Rice       2.25  1.01  0.47  5.00
Wheat      2.27  1.03  0.45  4.97


Cell 6 — Save clean version to PostgreSQL

In [29]:
# Use .env for password — no hardcoding
import os
from dotenv import load_dotenv
load_dotenv()
from sqlalchemy import create_engine
import time

def write_to_postgres(df, table_name, engine):
    try:
        start = time.time()
        
        df.to_sql(
            name    = table_name,
            con     = engine,
            if_exists = "replace",
            index   = False,
            chunksize = 1000
        )
        
        result= pd.read_sql(
            f"SELECT COUNT(*) as count FROM {table_name}",
            engine
        )
        db_count = result["count"][0]
        
        elapsed = time.time() - start
        
        print(f"Written to {table_name}")
        print(f"Rows written: {len(df):,}")
        print(f"Rows in DB: {db_count:,}")
        print(f"Time elapsed: {elapsed:.2f}s")
    
    except Exception as e:
        print(f"Failed to write to database: {e}")

engine = create_engine(os.getenv("DATABASE_URL"))
write_to_postgres(df, "climate_crops", engine)
write_to_postgres(df, "ca_crops", engine)

Written to climate_crops
Rows written: 10,000
Rows in DB: 10,000
Time elapsed: 1.08s
Written to ca_crops
Rows written: 10,000
Rows in DB: 10,000
Time elapsed: 0.91s
